# MI-GAN Kitchen Fine-Tuning (Round 2)

Builds on the Places365 boundary-fixed model and trains on Josh's
kitchen photos. Starts from `migan_finetuned.pt` (boundary fix baked in)
and adds kitchen-specific knowledge on top.

**How to use:**
1. H100 or A100 GPU + High-RAM (Runtime > Change runtime type)
2. Run cells top to bottom
3. Cell 3 will ask you to upload `migan_finetuned.pt` from your Mac
4. Cell 5 trains for 20K steps
5. Cell 7 downloads the result

## Cell 1: Setup

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!pip install -q gdown lpips coremltools Pillow scikit-image pyspng imgaug ninja

import os
if not os.path.exists('/content/migan-finetune'):
    !git clone https://github.com/joscraw/migan-finetune.git /content/migan-finetune
else:
    !cd /content/migan-finetune && git pull

os.chdir('/content/migan-finetune')
print(f'Working directory: {os.getcwd()}')

## Cell 2: Download Kitchen Photos

In [ ]:
import os

kitchen_dir = '/content/kitchen_photos'
os.makedirs(kitchen_dir, exist_ok=True)

# Download kitchen photos from Dropbox
zip_path = '/content/kitchen_photos.zip'
if not os.path.exists(zip_path):
    print('Downloading kitchen photos from Dropbox...')
    !wget -q -O {zip_path} "https://www.dropbox.com/scl/fi/15t8akmjy71xq29ku0uyx/kitchen_photos.zip?rlkey=w6gfvg1d3qz0mv4uk30qnxoef&st=f3kzswnq&dl=1"
    print('Downloaded!')
else:
    print('Kitchen photos zip already downloaded')

# Unzip
print('Unzipping...')
!unzip -q -o {zip_path} -d /content/kitchen_photos/

# Handle nested folder if zip contains a subfolder
nested = '/content/kitchen_photos/kitchen photos'
if os.path.exists(nested):
    !mv "/content/kitchen_photos/kitchen photos"/* /content/kitchen_photos/
    !rm -rf "/content/kitchen_photos/kitchen photos"

# Count
num = len([f for f in os.listdir(kitchen_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'Kitchen photos: {num}')

## Cell 3: Load Boundary-Fixed Weights + Discriminator

Upload `migan_finetuned.pt` from your Mac when prompted.
This is the Places365 boundary-fixed model — we build on top of it.

In [ ]:
import gdown
import pickle
import sys

sys.path.insert(0, '/content/migan-finetune')
os.makedirs('pretrained', exist_ok=True)

# Upload the boundary-fixed weights from your Mac
finetuned_path = 'pretrained/migan_finetuned.pt'
if not os.path.exists(finetuned_path):
    print('Upload your Places365 fine-tuned migan_finetuned.pt file:')
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        os.rename(name, finetuned_path)
    print(f'Saved to {finetuned_path}')
else:
    print(f'Fine-tuned weights already at {finetuned_path}')

# Download discriminator (.pkl)
pkl_path = 'pretrained/migan_512_unfused.pkl'
if not os.path.exists(pkl_path):
    print('Downloading discriminator...')
    gdown.download(id='1akS5165AOcAupDo8jbq9-eGlmBrRFMMd', output=pkl_path, quiet=False)
else:
    print(f'Discriminator already at {pkl_path}')

# Load discriminator
print('Loading discriminator...')
with open(pkl_path, 'rb') as f:
    checkpoint = pickle.Unpickler(f).load()
disc = checkpoint['D']
for p in disc.parameters():
    p.requires_grad_(True)
disc.train()
del checkpoint
print(f'Discriminator: {sum(p.numel() for p in disc.parameters()):,} params')

# Load generator from boundary-fixed weights
from finetune.train import load_pretrained
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_pretrained(finetuned_path, resolution=512, device=device)
print('Ready! (starting from boundary-fixed weights)')

## Cell 4: Measure Baseline

In [ ]:
from finetune.dataset import InpaintingDataset
from finetune.train import measure_boundary_gap, save_comparison
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Create dataset from kitchen photos
dataset = InpaintingDataset(kitchen_dir, resolution=512, hole_range=(0.15, 0.45))

# Measure baseline on kitchen images
print('Measuring boundary gap on kitchen images...')
test_images = []
test_masks = []
for i in range(min(20, len(dataset))):
    img, mask = dataset[i]
    test_images.append(img)
    test_masks.append(mask)

gap_before = measure_boundary_gap(model, test_images, test_masks, device)
print(f'\n>>> BOUNDARY GAP (kitchen baseline): {gap_before:.6f}')

# Show samples
os.makedirs('/content/results_kitchen/before', exist_ok=True)
for i in range(min(5, len(test_images))):
    save_comparison(model, test_images[i], test_masks[i],
                    f'/content/results_kitchen/before/sample_{i}.png', device)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i in range(min(3, len(test_images))):
    img = Image.open(f'/content/results_kitchen/before/sample_{i}.png')
    axes[i].imshow(img)
    axes[i].set_title(f'Kitchen sample {i}: masked | full output | composited')
    axes[i].axis('off')
plt.suptitle(f'Kitchen Baseline (boundary-fixed model) — Gap: {gap_before:.6f}', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 5: Train on Kitchen Photos (20K steps)

Builds on boundary-fixed weights. Rebalanced loss weights:
- Reconstruction: 2.0 (good fills)
- Identity: 5.0 (maintain boundary fix)
- Boundary: 2.0 (maintain boundary fix)
- Learning rate: 1e-4 (gentle fine-tuning)

In [ ]:
from finetune.train import train

model = train(
    model=model,
    disc=disc,
    dataset=dataset,
    output_dir='/content/results_kitchen/training',
    num_steps=20000,
    batch_size=4,
    g_lr=1e-4,
    d_lr=1e-4,
    lambda_reconstruction=2.0,
    lambda_identity=5.0,
    lambda_boundary=2.0,
    lambda_adversarial=1.0,
    lambda_perceptual=0.1,
    r1_gamma=10.0,
    d_reg_every=16,
    use_perceptual=True,
    save_every=5000,
    log_every=100,
    device=device,
)

print('\nKitchen training complete!')

## Cell 6: Evaluate Results

In [ ]:
print('Measuring boundary gap (AFTER kitchen training)...')
gap_after = measure_boundary_gap(model, test_images, test_masks, device)

print(f'\n{"="*50}')
print(f'RESULTS')
print(f'{"="*50}')
print(f'Boundary gap BEFORE:  {gap_before:.6f}')
print(f'Boundary gap AFTER:   {gap_after:.6f}')
print(f'{"="*50}')

os.makedirs('/content/results_kitchen/after', exist_ok=True)
for i in range(min(5, len(test_images))):
    save_comparison(model, test_images[i], test_masks[i],
                    f'/content/results_kitchen/after/sample_{i}.png', device)

fig, axes = plt.subplots(3, 2, figsize=(20, 18))
for i in range(min(3, len(test_images))):
    before_img = Image.open(f'/content/results_kitchen/before/sample_{i}.png')
    after_img = Image.open(f'/content/results_kitchen/after/sample_{i}.png')
    axes[i, 0].imshow(before_img)
    axes[i, 0].set_title('BEFORE kitchen training', fontsize=14)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(after_img)
    axes[i, 1].set_title('AFTER kitchen training', fontsize=14)
    axes[i, 1].axis('off')

plt.suptitle(f'Kitchen Training: gap {gap_before:.6f} → {gap_after:.6f}', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 7: Download Model

In [ ]:
from google.colab import files
files.download('/content/results_kitchen/training/migan_finetuned.pt')